In [1]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
from scipy import stats

In [2]:
PRICE_PER_MS_PER_MB_PER_100K_REQ = 0.00000162109
def get_estimated_cost(mem, time):
    # mem in MB, time in ms
    if mem < 128:
        mem = 128
    return mem * PRICE_PER_MS_PER_MB_PER_100K_REQ * time

In [3]:
# data processing
@dataclass
class Result:  # in the form of (mean, std)
    prep: tuple[float, float]
    init: tuple[float, float]
    exec: tuple[float, float]
    inport: tuple[float, float]
    e2e: tuple[float, float]
    memory: tuple[float, float]
    cost: tuple[float, float]
    billed: tuple[float, float]


def filter_outlier(df, z_score=1):
    # Filter outliers rows based on e2e_latency
    z = np.abs(stats.zscore(df["e2e_latency"]))
    return df[(z < z_score)]

def get_phase_times_from_csv(file_path: str, z_score: int = 1):
    df = filter_outlier(pd.read_csv(file_path), z_score)

    import_times = df["import_time"]
    init_times = df["billed_duration"] - df["duration"]
    prep_times = 1000 * df["e2e_latency"] - df["billed_duration"]

    billed_duration = df["billed_duration"]
    billed_mean = billed_duration.mean()
    billed_std = billed_duration.std()

    prep_mean = prep_times.mean()
    prep_std = prep_times.std()
    init_mean = init_times.mean()
    init_std = init_times.std()
    exec_mean = df["duration"].mean()
    exec_std = df["duration"].std()
    import_mean = import_times.mean()
    import_std = import_times.std()

    e2e_mean = df["e2e_latency"].mean()
    e2e_std = df["e2e_latency"].std()

    mem_mean = df["mem_used"].mean()
    mem_std = df["mem_used"].std()

    
    return Result(
        prep=(prep_mean, prep_std),
        init=(init_mean, init_std),
        exec=(exec_mean, exec_std),
        inport=(import_mean, import_std),
        e2e=(e2e_mean, e2e_std),
        memory=(mem_mean, mem_std),
        billed=(billed_mean, billed_std),
        cost=(get_estimated_cost(mem_mean, billed_mean), get_estimated_cost(mem_std, billed_std)),
    )

In [4]:
def compare_difference(a: Result, b: Result):
    df = pd.DataFrame(
        {
            "prep": [(b.prep[0] - a.prep[0]) / a.prep[0]],
            "init": [(b.init[0] - a.init[0]) / a.init[0]],
            "exec": [(b.exec[0] - a.exec[0]) / a.exec[0]],
            "import": [(b.inport[0] - a.inport[0]) / a.inport[0]],
            "e2e": [(b.e2e[0] - a.e2e[0]) / a.e2e[0]],
            "memory": [(b.memory[0] - a.memory[0]) / a.memory[0]],
            "billed": [(b.billed[0] - a.billed[0]) / a.billed[0]],
            "cost": [(b.cost[0] - a.cost[0]) / a.cost[0]],
        })
    return df

In [21]:
def generate_csv(apps: list[str], debloat: bool, z_score: int):
    import_time = []
    exec_time = []
    e2e_latency = []
    billed_duration = []
    memory = []
    cost = []

    old_measurements = pd.read_csv(OUTPUT_DIR / Path("measurements_old.csv"))

    dir = ORIGINAL_DIR if not debloat else DEBLOATED_DIR

    for app in apps:
        file_name = f"{app}.csv" if not debloat else f"{app}-debloat.csv"
        # If file doesn't exits, use old mearuements
        if not (dir / file_name).exists():
            print(f"{dir / file_name} not found, using old measurements")
            import_time.append(old_measurements[old_measurements["app"] == app]["import_time"].values[0])
            exec_time.append(old_measurements[old_measurements["app"] == app]["exec_time"].values[0])
            e2e_latency.append(old_measurements[old_measurements["app"] == app]["e2e_latency"].values[0])
            billed_duration.append(old_measurements[old_measurements["app"] == app]["billed_duration"].values[0])
            memory.append(old_measurements[old_measurements["app"] == app]["memory"].values[0])
            cost.append(old_measurements[old_measurements["app"] == app]["cost"].values[0])
            continue
        result = get_phase_times_from_csv(file_name, dir, z_score=z_score)
        # Convert from ms to seconds and keep 2 decimal
        import_mean = round(result.inport[0], 2)
        exec_mean = round(result.exec[0] / 1000, 2)
        e2e_mean = round(result.e2e[0], 2)
        billed = round(result.billed[0] / 1000, 2)
        mem_mean = round(result.memory[0], 2)
        import_time.append(import_mean)
        exec_time.append(exec_mean)
        e2e_latency.append(e2e_mean)
        billed_duration.append(billed)
        memory.append(mem_mean)
        cost.append(round(result.cost[0],2))
    
    df = pd.DataFrame({
        "app": apps,
        "import_time": import_time,
        "exec_time": exec_time,
        "e2e_latency": e2e_latency,
        "billed_duration": billed_duration,
        "memory": memory,
        "cost": cost
    })
    return df

In [40]:
compare_difference(get_phase_times_from_csv("baseline_warm/huggingface.csv"), get_phase_times_from_csv("debloat_k10_warm/huggingface-k9.csv"))

,prep,init,exec,import,e2e,memory,billed,cost
0,-0.063578,-0.088171,0.026041,-0.316936,0.001853,-0.094336,0.025767,-0.071


In [50]:
compare_difference(get_phase_times_from_csv("baseline_warm/lightgbm.csv"), get_phase_times_from_csv("debloat_k10_warm/lightgbm-k10.csv"))

,prep,init,exec,import,e2e,memory,billed,cost
0,0.133687,-0.223994,-0.060353,-0.481196,0.115151,-0.232625,-0.072493,-0.072493


In [36]:
compare_difference(get_phase_times_from_csv("baseline_warm/qiskit-nature.csv"), get_phase_times_from_csv("debloat_k10_warm/qiskit-nature-k10.csv"))

,prep,init,exec,import,e2e,memory,billed,cost
0,0.021727,0.042341,0.015698,-0.009477,0.01854,-0.004904,0.015841,0.010859


In [18]:
compare_difference(get_phase_times_from_csv("baseline_warm/shapely-numpy.csv"), get_phase_times_from_csv("debloat_k10_warm/shapely-numpy-debloat.csv"))

,prep,init,exec,import,e2e,memory,billed,cost
0,-0.154343,0.804611,-0.17382,-0.248471,-0.151318,-0.114628,-0.037962,-0.037962


In [51]:
compare_difference(get_phase_times_from_csv("baseline_warm/pandas.csv"), get_phase_times_from_csv("debloat_k10_warm/pandas-k10.csv"))

,prep,init,exec,import,e2e,memory,billed,cost
0,0.372546,0.24006,-0.828637,-0.017653,-0.009487,-0.106498,-0.81545,-0.81545


In [46]:
compare_difference(get_phase_times_from_csv("baseline_warm/igraph.csv"), get_phase_times_from_csv("debloat_k10_warm/igraph-k10.csv"))

,prep,init,exec,import,e2e,memory,billed,cost
0,-0.001148,0.421386,0.008142,-0.268676,0.002622,-0.034965,0.032353,0.032353


In [5]:
compare_difference(get_phase_times_from_csv("baseline312/tensorflow312.csv"), get_phase_times_from_csv("debloat_k10_with_fallback/tensorflow-k10.csv"))

,prep,init,exec,import,e2e,memory,billed,cost
0,0.451828,-0.119465,5.677655,-0.122095,-0.080264,-0.075875,-0.109607,-0.177165


In [38]:
huggingface_baseline = get_phase_times_from_csv("baseline/huggingface.csv")
huggingface_baseline

Result(prep=(np.float64(282.984661003978), np.float64(28.65100190190403)), init=(np.float64(5264.301030927834), np.float64(380.0969361406501)), exec=(np.float64(493.24536082474225), np.float64(35.47352495940504)), inport=(np.float64(3.2312804104126607), np.float64(0.2615080385923333)), e2e=(np.float64(6.040531052756555), np.float64(0.41069936568392945)), memory=(np.float64(840.6082474226804), np.float64(7.349258325718611)), cost=(np.float64(7.845817817228166), np.float64(0.08280186393922094)), billed=(np.float64(5757.546391752578), np.float64(399.04605051241055)))

In [39]:
huggingface_k10 = get_phase_times_from_csv("debloat_k10/huggingface-debloat.csv")
huggingface_k10 

Result(prep=(np.float64(285.94717560351734), np.float64(25.924038635387653)), init=(np.float64(4920.504647887324), np.float64(130.4457430924661)), exec=(np.float64(73.32633802816902), np.float64(5.188053100134858)), inport=(np.float64(2.974122356361067), np.float64(0.10449186634637968)), e2e=(np.float64(5.27977816151901), np.float64(0.1349538859774088)), memory=(np.float64(613.7183098591549), np.float64(0.4530247105070317)), cost=(np.float64(4.968325568093816), np.float64(0.027490537057364033)), billed=(np.float64(4993.830985915493), np.float64(132.48482241001827)))

In [40]:
huggingface_fallback = get_phase_times_from_csv("fallback/huggingface-k10.csv")

In [41]:
compare_difference(huggingface_baseline, huggingface_k10)

,prep,init,exec,import,e2e,memory,billed,cost
0,0.010469,-0.065307,-0.851339,-0.079584,-0.125941,-0.269912,-0.132646,-0.366755


In [42]:
compare_difference(huggingface_baseline, huggingface_fallback)

,prep,init,exec,import,e2e,memory,billed,cost
0,0.014976,-0.074302,0.153735,-0.004224,-0.051499,-0.254719,-0.054766,-0.295535


In [44]:
compare_difference(huggingface_k10, huggingface_fallback)

,prep,init,exec,import,e2e,memory,billed,cost
0,0.004461,-0.009623,6.760847,0.081876,0.085169,0.02081,0.08979,0.112469


In [5]:
baseline = get_phase_times_from_csv("baseline/huggingface.csv")
debloat_c = get_phase_times_from_csv("debloat_k10/huggingface-debloat.csv")
debloat_w = get_phase_times_from_csv("warm_k10/huggingface-k10.csv")
debloat_fallback_c_c = get_phase_times_from_csv("fallback_c_c/huggingface-k10.csv")
debloat_fallback_c_w = get_phase_times_from_csv("fallback_c_w/huggingface-k10.csv")
debloat_fallback_w_c = get_phase_times_from_csv("fallback_w_c/huggingface-k10.csv")
debloat_fallback_w_w = get_phase_times_from_csv("fallback_w_w/huggingface-k10.csv")

In [7]:
df = pd.DataFrame({
    'Category': ['Baseline', 'debloat_c', 'debloat_w', 'debloat_fallback_c_c', 'debloat_fallback_c_w', 'debloat_fallback_w_c', 'debloat_fallback_w_w'],
    'Memory (principal only)': [baseline.memory[0], debloat_c.memory[0], debloat_w.memory[0], debloat_fallback_c_c.memory[0], debloat_fallback_c_w.memory[0], debloat_fallback_w_c.memory[0], debloat_fallback_w_w.memory[0]],
    'e2e_latency': [baseline.e2e[0], debloat_c.e2e[0], debloat_w.e2e[0], debloat_fallback_c_c.e2e[0], debloat_fallback_c_w.e2e[0], debloat_fallback_w_c.e2e[0], debloat_fallback_w_w.e2e[0]],
    'Cost (Principal only)': [baseline.cost[0], debloat_c.cost[0], debloat_w.cost[0], debloat_fallback_c_c.cost[0], debloat_fallback_c_w.cost[0], debloat_fallback_w_c.cost[0], debloat_fallback_w_w.cost[0]],
})
df

,Category,Memory (principal only),e2e_latency,Cost (Principal only)
0,Baseline,840.608247,6.040531,7.845818
1,debloat_c,613.718310,5.279778,4.968326
2,debloat_w,653.464646,0.074853,0.006912
3,debloat_fallback_c_c,626.514286,11.732486,11.621357
4,debloat_fallback_c_w,626.489583,5.729452,5.527106
5,debloat_fallback_w_c,629.140845,6.167183,6.216368
6,debloat_fallback_w_w,630.000000,0.321839,0.258103


In [8]:
jsym_baseline = get_phase_times_from_csv("baseline312/jsym312.csv")
jsym_baseline

Result(prep=(np.float64(245.31358353296915), np.float64(23.0585708920165)), init=(np.float64(707.90625), np.float64(52.812736258326446)), exec=(np.float64(266.34375), np.float64(10.627527296773314)), inport=(np.float64(0.5524510112073686), np.float64(0.03686522724311922)), e2e=(np.float64(1.2195635835329692), np.float64(0.06545072582121397)), memory=(np.float64(97.23611111111111), np.float64(0.4276715718818953)), cost=(np.float64(0.20215640735999998), np.float64(0.01208629670156638)), billed=(np.float64(974.25), np.float64(58.247347760449664)))

In [14]:
jsym_zip = get_phase_times_from_csv("baseline_zip/jsym-snap.csv")
jsym_zip

Result(prep=(np.float64(1191.1083510525254), np.float64(24.47000273350832)), init=(np.float64(0.4843373493975902), np.float64(0.2907735384473297)), exec=(np.float64(303.8409638554217), np.float64(6.61218767798688)), inport=(np.float64(0.6661199316921005), np.float64(0.01756516045584104)), e2e=(np.float64(1.4954336522573448), np.float64(0.02778626630440505)), memory=(np.float64(92.0), np.float64(0.0)), cost=(np.float64(0.06314735392385541), np.float64(0.0013794858778829438)), billed=(np.float64(304.32530120481925), np.float64(6.648140091518977)))

In [9]:
jsym_snap = get_phase_times_from_csv("snap_start_results/jsym-snap.csv")
jsym_snap

Result(prep=(np.float64(1180.8466781018728), np.float64(31.693882550663115)), init=(np.float64(0.45337349397590426), np.float64(0.30088194390310335)), exec=(np.float64(303.2695180722891), np.float64(10.278390058950757)), inport=(np.float64(0.6630476210490767), np.float64(0.027814792357288207)), e2e=(np.float64(1.4845695696681378), np.float64(0.038559755369255835)), memory=(np.float64(92.0), np.float64(0.0)), cost=(np.float64(0.06302235421301204), np.float64(0.00213371008418006)), billed=(np.float64(303.72289156626505), np.float64(10.282963951820516)))

In [11]:
jsym_debloat = get_phase_times_from_csv("debloat_k10/jsym-debloat.csv")
jsym_debloat

Result(prep=(np.float64(246.45283223320456), np.float64(39.58045458019028)), init=(np.float64(630.6798823529411), np.float64(45.743263893743546)), exec=(np.float64(317.2495294117647), np.float64(14.443361459580691)), inport=(np.float64(0.4889693484586828), np.float64(0.033836646637890884)), e2e=(np.float64(1.1943822439979104), np.float64(0.0590694767274297)), memory=(np.float64(82.0), np.float64(0.0)), cost=(np.float64(0.19669489793505882), np.float64(0.011513921506969043)), billed=(np.float64(947.9294117647058), np.float64(55.48890670671934)))

In [ ]:
compare_difference(jsym_baseline, jsym_snap)

,prep,init,exec,import,e2e,memory,billed,cost
0,3.813621,-0.99936,0.13864,0.200193,0.217296,-0.053849,-0.68825,-0.68825


In [13]:
compare_difference(jsym_debloat, jsym_snap)

,prep,init,exec,import,e2e,memory,billed,cost
0,3.79137,-0.999281,-0.044066,0.356011,0.24296,0.121951,-0.679593,-0.679593


In [15]:
compare_difference(jsym_zip, jsym_snap)

,prep,init,exec,import,e2e,memory,billed,cost
0,-0.008615,-0.06393,-0.001881,-0.004612,-0.007265,0.0,-0.001979,-0.001979
